# Exercise 6..: training a Field-of-Experts regularizer
In this exercise, we will train the Field-of-Experts regularizer for

- an image denoising problem and,
- apply it for tomographic reconstruction.

## Change runtime type and install DeepInverse package
First, we install the deepinverse library. Remeber to change the runtime of this notebook select ``Runtime`` -> ``Change runtime type`` and pick an available hardware accelerator.

In [ ]:
!pip install git+https://github.com/deepinv/deepinv.git#egg=deepinv

In [ ]:
# Set tot true, if you want to use Weights&Biases logger
save_to_wandb = False

if save_to_wandb:
    import wandb
    wandb.login()

In [ ]:
import deepinv as dinv
from torch.utils.data import DataLoader
import torch
from pathlib import Path
from torchvision import transforms
from deepinv.utils.demo import load_dataset

import matplotlib.pyplot as plt

# Setup paths for data loading and checkpoints

In [ ]:
BASE_DIR = Path(".")
DATA_DIR = BASE_DIR / "measurements"
CKPT_DIR = BASE_DIR / "ckpts"

# Set the global random seed from pytorch to ensure reproducibility of the example.
torch.manual_seed(0)

device = dinv.utils.get_freer_gpu() if torch.cuda.is_available() else "cpu"

# Load base image datasets and degradation operators
In this example, we use the BSDS5000 dataset for training and the CBSD68 dataset for testing. We work with images of size 64x64.

To split BSDS5000 into training and validation datasets
1. Use `torch.randperm` https://docs.pytorch.org/docs/stable/generated/torch.randperm.html to generate random permutation of indices.
2. Use `torch.utils.data.Subset` https://docs.pytorch.org/docs/stable/data.html#torch.utils.data.Subset to select a subset of the dataset.

To make images grayscale, we use [Grayscale transform](https://docs.pytorch.org/vision/0.22/generated/torchvision.transforms.v2.Grayscale.html)

In [ ]:
from torchvision.transforms import (
     RandomCrop,
     RandomVerticalFlip,
     Compose,
     RandomHorizontalFlip,
     CenterCrop,
     RandomApply,
     RandomRotation,
     v2
)

operation = "denoising"
train_dataset_name = "CBSD68"
test_dataset_name = "set3c"

img_size = 64 if torch.cuda.is_available() else 32

grayscale = True

if grayscale:
    val_transform = transforms.Compose(
        [v2.Grayscale(), transforms.CenterCrop(img_size), transforms.ToTensor()]
    )
    
    test_transform = transforms.Compose(
        [v2.Grayscale(), transforms.CenterCrop(320), transforms.ToTensor()]
    )
    
    train_transform = transforms.Compose(
        [v2.Grayscale(), transforms.RandomCrop(img_size),
         RandomHorizontalFlip(p=0.5),
         RandomVerticalFlip(p=0.5),
         RandomApply([RandomRotation((90, 90))], p=0.5),
         transforms.ToTensor()]
    )
else:
    val_transform = transforms.Compose(
        [transforms.CenterCrop(img_size), transforms.ToTensor()]
    )
    
    test_transform = transforms.Compose(
        [transforms.CenterCrop(320), transforms.ToTensor()]
    )
    
    train_transform = transforms.Compose(
        [transforms.RandomCrop(img_size),
         RandomHorizontalFlip(p=0.5),
         RandomVerticalFlip(p=0.5),
         RandomApply([RandomRotation((90, 90))], p=0.5),
         transforms.ToTensor()]
    )

DATASET_DIR = BASE_DIR / ".." / "pnp" / "datasets" / "BSDS500"
train_dataset = dinv.datasets.BSDS500(DATASET_DIR, download=True, train = True, transform=train_transform)
val_dataset = dinv.datasets.BSDS500(DATASET_DIR, download=True, train = True, transform=val_transform)
DATASET_DIR_CBSD = BASE_DIR / ".." / "pnp" / "datasets" / "CBSD68"

# Uncomment the next line and comment the last to swith to center crops on the test set
test_dataset = dinv.datasets.CBSD68(root = DATASET_DIR_CBSD, download=True, transform = test_transform, rotate = False)
indices = torch.randperm(len(train_dataset))
val_size = 40
train_dataset = FILL_THIS_PART
val_dataset = FILL_THIS_PART

# Plot an image from a dataset
Try plotting it multiple times! Why does it produce different images for the training set?
Read about data augmentation and transforms at https://docs.pytorch.org/vision/0.9/transforms.html and https://deepinv.github.io/deepinv/user_guide/training/transforms.html#transform.

In [ ]:
plt.imshow(train_dataset[0].permute(1, 2, 0), cmap = 'gray')
plt.axis("off")
plt.show()

plt.imshow(val_dataset[0].permute(1, 2, 0),cmap = 'gray')
plt.axis("off")
plt.show()

plt.imshow(test_dataset[0].permute(1, 2, 0),cmap = 'gray')
plt.axis("off")
plt.show()

# Set up physics
Define a physics object corresponding to denoising problem with Gaussian noise and standard deviation $\sigma = 25/255$. See [this overview](https://deepinv.github.io/deepinv/auto_examples/physics/demo_physics_tour.html) and a [complete list](https://deepinv.github.io/deepinv/api/deepinv.physics.html) of physics models available at DeepInverse.

In [ ]:
sigma = 25.0/255 # 0.05
physics = dinv.physics.Denoising(dinv.physics.GaussianNoise(sigma)) #())

# Prepare dataloaders
Dataloaders handle loading datasets into memory. You can set up the batchs size and whether you want the elements be randomly reschuffled, see Remark 5.3.4.

In [ ]:
# n_channels = 3 
num_workers = 1 if torch.cuda.is_available() else 1

train_batch_size = 32 if torch.cuda.is_available() else 1
val_batch_size = 20 if torch.cuda.is_available() else 1
test_batch_size = 4 if torch.cuda.is_available() else 1

train_dataloader = DataLoader(
    train_dataset, batch_size=train_batch_size, num_workers=num_workers, shuffle=True
)

val_dataloader = DataLoader(
    val_dataset, batch_size=val_batch_size, num_workers=num_workers, shuffle=False
)

test_dataloader = DataLoader(
    test_dataset, batch_size=test_batch_size, num_workers=num_workers, shuffle=False
)

# Set up the reconstruction network

We implement the Field-of-Experts regularizer are presented in the lecture. 

Note that a more advanced version replaces the Huber potential functions with learnable potentials can be found here. Learning the potential function significantly improves the FoE model's performance on denoising tasks (by around 2 PSNR). For the details, see
[Alexis Goujon, Sebastian Neumayer, Pakshal Bohra, Stanislas Ducotterd, and Michael Unser. 
A neural-network-based convex regularizer for inverse problems.
IEEE Transactions on Computational Imaging, 9:781–795, 2023](https://ieeexplore.ieee.org/abstract/document/10223264)
and
[Alexis Goujon, Sebastian Neumayer, and Michael Unser. 
Learning Weakly Convex Regularizers for Convergent Image-Reconstruction Algorithms. 
SIAM Journal on Imaging SciencesVol. 17, Iss. 1 (2024) 10.1137/23M1565243](https://epubs.siam.org/doi/abs/10.1137/23M1565243).
Their implementation is available at https://github.com/axgoujon/convex_ridge_regularizers

In [ ]:
from linear_spline import LinearSpline

class ZeroMean(torch.nn.Module):
    r"""
    This class enforces the constraint < X, \1 > = 0 over spatial dimensions, where X is the convolution filter of size ( input_channels, output_channels, kernel_dim, kernel_dim)
    In forward, implement an orthogonal projection onto the complement of the set {X : < X, \1 > = 0}
    """
    def forward(self, X):
        return X - X.mean(dim=(-2,-1))[(...,) + (None,) *2]

class FoE(torch.nn.Module):
    r"""
    Field-of-Experts regularizer.

    :param int channels: Number of input channels.
    :param int num_filters: Number of filters channels (Tensor or int).
    :param int kernel_dim: dimension of the convolutional kernels.
    :param bool norm_weights: Whether to normalize weights.
    :param float beta: Initial global scaling parameter in the Huber function, see equation (6.60).
    :param torch.device, str device: Device to put the model on.
    :param bool conv: if True: use Huber potential \psi^\beta as in (6.60), yielding convex regularizer,
                      if False: that \tilde \psi^\beta = \psi^\beta - \psi^1, giving nonconvex potential.
    :param int img_size: width of a square image. Needed to compute the spectral norm of the stacked convolution matrix.
    :param float alpha: Regularization parameter in 0.5 \|Ax - y\|_2^2 + \alpha R(x).
    :param bool alpha_learnable: Whether we want to optimize alpha 
    """

    def __init__(
        self,
        in_channels: int = 3,
        num_filters: int = 64,
        kernel_dim: int = 5,
        norm_weights: bool = False,
        beta : float = 0.1,
        device: torch.device | str = "cpu",
        conv : bool = True,
        img_size : int = 9,
        alpha : float = 1.0,
        alpha_learnable : bool = False,
    ):
        super(FoE, self).__init__()

        self.n_in_channels = in_channels
        self.n_filters = num_filters
        self.kernel_size = kernel_dim
        self.padding = (self.kernel_size - 1) // 2
            
        self.W = torch.nn.ModuleList(
            [torch.nn.Conv2d(self.n_in_channels, 4, self.kernel_size, bias=False, stride=1, padding=self.padding,padding_mode="zeros",device = device),
             torch.nn.Conv2d(4, 8, self.kernel_size, bias=False, stride=1, padding=self.padding,padding_mode="zeros",device = device),
             torch.nn.Conv2d(8, self.n_filters, self.kernel_size, bias=False, stride=1, padding=self.padding,padding_mode="zeros",device = device)
            ])

        for i in range(3):
            torch.nn.utils.parametrize.register_parametrization(self.W[i], "weight", ZeroMean())
            
        # scaling parameters of the potential function for each individual filter, see text above (6.60)
        self.alphaj = torch.nn.Parameter(torch.ones(self.n_filters).abs().to(device))
        self.alphaj.requires_grad_(True)
        
        self.alpha = torch.nn.Parameter(alpha*torch.ones(1).to(device))
        self.alpha.requires_grad_(alpha_learnable)
        self.beta = torch.nn.Parameter(beta * torch.ones(1).to(device))
        self.norm_weights = norm_weights
        self.device = device
        self.conv = conv
        self.img_size = img_size

        # We store the top eigenvector to reuse it in the power method
        self.eig_vec = torch.randn([1,in_channels, img_size,img_size]).to(device)
        self.eig_val = torch.ones(1).to(device)

    def forward(self, x: torch.Tensor,**kwarg) -> torch.Tensor:
         r"""
        Calculate potential function of the FoE.

        :param torch.Tensor x: Input tensor of shape ``(B, C, H, W)``.
        """

         # Apply stacked convolution layers
        z = self.W[0](x)
        z = self.W[1](z)
        z = self.W[2](z)

        # Apply scaling factors per channel dimension
        z = z*self.alphaj.abs()[ (None,) *1 + (...,) + (None,) *2]
        # Compute activation
        z = self.activation(x)
        
        # Apply scaling factors per channel dimension
        z = z / self.alphaj.square()[  (None,) *1 + (...,) + (None,) *2]

        # Multiply with regularization parameter. Again, we use a safeguard here to make sure it is positive
        return (self.alpha.abs() + 1e-6)*z.sum(dim=(1,2,3))

    def activation(self,x):
        r"""
        Compute the action of the activation.
        :param torch.Tensor x: Input tensor of shape ``(B, C, H, W)``.
        """
        if not self.conv:
            # Compute \psi^1.
            zsq = x.square()
            zmax = x.abs().maximum(torch.ones([1]).to(device))
            z_one = zsq / zmax
            zsq = zsq.minimum(torch.ones([1]).to(device))
            z_one = z_one - 0.5*zsq

        # Compute \psi^\beta. Here we use a safeguard to avoid zero values
        beta = self.beta.clamp(min = 0) + 1e-6      
        z1 = x.square()
        z1 = z1/ x.abs().maximum(1.0/beta) 
        z = - 0.5*(beta*x.square()).minimum(1.0/beta)
        z = z.add(z1)
        
        if not self.conv:
            z = z.add(-z_one)

        return z
            
    def activation_grad(self,x):
        r"""
        Compute the action of the activation's gradient.
        :param torch.Tensor x: Input tensor of shape ``(B, C, H, W)``.
        """
        if not self.conv:
            z_one = FILL_THIS_PART
            z_one = FILL_THIS_PART

        beta = self.beta.clamp(min = 0) + 1e-6      
        z = FILL_THIS_PART
        z = FILL_THIS_PART
        
        if not self.conv:
            z = z.add(-z_one)

        return z        
        
    def grad(self, x: torch.Tensor,**kwarg) -> torch.Tensor:
        r"""
        Calculate the gradient of the potential function.
        :param torch.Tensor x: Input tensor of shape ``(B, C, H, W)``.
        """
        z = x
        
        for i in range(3):
            z = FILL_THIS_PART

        z = z*self.alphaj.abs()[ (None,) *1 + (...,) + (None,) *2]
        z = FILL_THIS_PART
        z = z / self.alphaj.abs()[  (None,) *1 + (...,) + (None,) *2]
        
        for i in torch.arange(2,-1,-1):
            z = torch.nn.functional.conv_transpose2d(z, FILL_THIS_PART, padding=self.W[i].padding, groups=self.W[i].groups, dilation=self.W[i].dilation)
            
        return z

    def update_eigen(self, num_it = 50):
        r"""
        Perform num_it iterations of the power method as in (6.59).
        
        :param torch.Tensor x: Input tensor of shape ``(B, C, H, W)``.
        """
        with torch.no_grad():
            z = self.eig_vec

            # Create transposes of convolutions
            W_adj = torch.nn.ModuleList(
                [torch.nn.ConvTranspose2d(4,self.n_in_channels, self.kernel_size, bias=False, stride=1, padding=self.padding,padding_mode="zeros",device = self.device),
                 torch.nn.ConvTranspose2d(8,4, self.kernel_size, bias=False, stride=1, padding=self.padding,padding_mode="zeros",device = self.device),
                 torch.nn.ConvTranspose2d(self.n_filters,8, self.kernel_size, bias=False, stride=1, padding=self.padding,padding_mode="zeros",device = self.device)
                ])
            
            for i in range(3):
                with torch.no_grad():
                    _ = W_adj[i].weight.copy_ (self.W[i].weight)

            # Implement power iterations
            for t in range(num_it):
                for i in range(3):
                    z = FILL_THIS_PART
            
                for i in torch.arange(2,-1,-1):
                    z = FILL_THIS_PART
                
                self.eig_val = z.norm().sqrt().detach()
                z = FILL_THIS_PART

            # Implement normalize each layer equally
            for i in range(3):
                self.W[i].weight.data /= FILL_THIS_PART
                
            self.eig_vec = z.detach()


    def set_image_size(self, img_size):
         r"""
        Normalization depends on the image size. Since we train on 64x64 images and test on 320x320, we need to renormalize
        
        :param torch.Tensor x: Input tensor of shape ``(B, C, H, W)``.
        """
        self.img_size = img_size
        self.update_eigen(100)

In [ ]:
class Solver(torch.nn.Module):
    def __init__(self,T, K,regularizer, init = 'y', alpha_decay = 1.0, tr = 1e-4):
        r"""
        This class minimizes 0.5 \|Ax - y\|_2^2 + \alpha R(x) with accelerated gradient descent.
    
        :param int T: Total number of iterations.
        :param int K: Number of iterations for trunctated backpropagation.
        :param nn.Model reg: Regularizer \alpha R(x).
        :param torch.device, str device: Device to put the model on.
        :param str init: if 'random': x^0 is randomly initialized, if 'pinv' x^0 = A^\dagger y and otherwise x^0 = y.
        :param float physics_spectral: spectral norm of the physics operator A
        :param float tr: relative residual threshold for the norm of consecutive iterates of accelerated gradient descent 
        """
        super().__init__()
        self.T=T
        self.K=K
        self.reg = regularizer
        self.device = regularizer.device
        self.init = init
        self.alpha_decay = alpha_decay
        self.last_epoch = 0
        self.tr = tr
        self.update_normalization(True, 500)
        self.data_fidelity = dinv.optim.data_fidelity.L2()
        self.physics_spectral = 1.0

    def ADG_it(self,x0, y, physics, num_it, lr, prev_num_it = 0, tr = 1e-4):
        r"""
        This function performs num_it iterations of accelerated gradient descent with given x^0, measurements y, physics A, and learning rate lr.
        Parameter prev_num_it is used to set the value of theta appropriately when the function is called the second time.
        tr is the relative residual threshold for the norm of consecutive iterates. 
        """
        
        x = x0
        z = x0.clone()
        data_fidelity = dinv.optim.data_fidelity.L2()

        for k in range(num_it):
            x_old = x.clone()
            theta = 2.0/(2 + k + prev_num_it)
            u = FILL_THIS_PART
            grad = FILL_THIS_PART
            grad = grad.add(self.reg.grad(u))
            x = FILL_THIS_PART
            z = FILL_THIS_PART

            if ((x - x_old).square().sum(dim=(1,2,3)).sqrt() / x_old.square().sum(dim=(1,2,3)).sqrt()).max() < tr:
                break
            
        return x

    def forward(self,
        y: torch.Tensor,
        physics: dinv.physics.Physics,
        train: bool = False,
        **kwargs,
    ) -> torch.Tensor:        
        r"""
        Find a solution of the optimization problem argmin 0.5 \|Ax - y\|_2^2 + \alpha R(x)
        """
        
        if self.reg.conv:
            beta = self.reg.beta.clamp(min=0) + 1e-6
        else:
            beta = max(self.reg.beta.clamp(min=0) + 1e-6,1)
            
        if self.reg.norm_weights:   
            stepsize = FILL_THIS_PART
        else:
            stepsize = FILL_THIS_PART
            
        if self.init == 'random':
            init = torch.randn(size=y.shape).to(self.device)
        elif self.init == 'pinv':
            init = physics.A_dagger(y)
        else:
            init = y

        with torch.no_grad():
            x_hat = self.ADG_it(FILL_THIS_PART) 
        
        x_hat = x_hat.clone(memory_format=torch.contiguous_format).detach()

        if train:
            x_hat.requires_grad_(True)
            y.requires_grad_(True)
        
        x_hat2 = self.ADG_it(FILL_THIS_PART)
        
        return x_hat2

    def update_normalization(self, train, num_it=50):
        r"""
        We want to update the normalization of the weights regularly 
        """
        if train and self.reg.norm_weights:
            self.reg.update_eigen(num_it)

In [ ]:
if grayscale:
    reg = FoE(
        in_channels=1, num_filters=64, kernel_dim = 5, device = device, conv = True, beta = 1e3, img_size = img_size, norm_weights = True, alpha = 1e0, alpha_learnable = True
    )
else:
    reg = FoE(
        in_channels=3, num_filters=64, kernel_dim = 5, device = device, conv = True, beta = 1e2, img_size = img_size, norm_weights = True, alpha = 1e0, alpha_learnable = False)


print(reg.parameters)

solver = Solver(50,5,reg, init='y', alpha_decay = 0.87, tr = 1e-4)
print(solver.reg.alphaj)
print(solver.reg.alpha)


# Set up a Trainer
Trainer is a ready-made class for training a network. It takes model, physics, dataloaders, and training parameters as input and trains the network accordingly. Try different numbers of epochs and learning rates to see which works best. Note: Given the limited free GPU access on Colab, going beyond 50 epochs is not recommended. 

See https://deepinv.github.io/deepinv/user_guide/training/trainer.html#trainer
and https://deepinv.github.io/deepinv/api/stubs/deepinv.Trainer.html for more details.

In [ ]:
# We need to include weight normalization in the training. Before running the step for each batch, and after the last batch, to update it before validation 
class Bilevel_Trainer(dinv.Trainer):
    def step(
        self,
        epoch,
        progress_bar,
        train_ite=None,
        train=True,
        last_batch=False,
        update_progress_bar=False,
    ):
        if train:
            self.model.update_normalization(train)
        
        super().step(epoch,progress_bar,train_ite,train,last_batch,update_progress_bar)

        if last_batch and not train:
            self.model.update_normalization(train)

In [ ]:
verbose = True  # print training information
wandb_vis = False  # plot curves and images in Weight&Bias

epochs = 50 # choose training epochs

if grayscale:
    learning_rate = FILL_THIS_PART
else:
    learning_rate = FILL_THIS_PART

# choose training losses
losses = [dinv.loss.SupLoss(metric=dinv.metric.MSE())]#, ZeroSumLoss()]

# choose optimizer and scheduler
optimizer = torch.optim.Adam(solver.reg.parameters(), lr=learning_rate, weight_decay=1e-8)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer,step_size=epochs,gamma=0.99) #step_size=1, gamma=0.1**(1.0/epochs))

if save_to_wandb:
    wandb_vis = True  # plot curves and images in Weight&Bias
    wandb_setup = {
        # Set the wandb entity where your project will be logged (generally your team name).
        "entity" : "oleh-melnyk-technical-university-of-berlin",
        # Set the wandb project where this run will be logged.
        "project" : "FoE-training",
        # Track hyperparameters and run metadata.
        "config" : {
            "learning_rate": learning_rate,
            "architecture": "FoE",
            "dataset": "CBSD68",
            "epochs": epochs,
            "scheduler_step_size": epochs,
            "sigma" : sigma,
            "alpha" : reg.alpha.cpu().item(),
            "kernel_dim" : reg.kernel_size,
            "num_filters" : reg.n_filters,
            "weight_decay" : 1e-8,
            "T" : solver.T,
            "K" : solver.K,
            "img_size" : img_size,
            "norm_weights": reg.norm_weights,
            "init" : solver.init,
            "conv" : solver.reg.conv,
            "grayscale" : grayscale,
            "beta" : reg.beta.cpu().item(),
            "tr" : solver.tr,
        },
    }
else:
    wandb_vis = False
    wandb_setup = None

trainer = Bilevel_Trainer(
    solver,
    device=device,
    save_path=str(CKPT_DIR / operation), 
    verbose=verbose,
    wandb_vis=wandb_vis,
    wandb_setup = wandb_setup,
    physics=physics,
    epochs=epochs,
    scheduler=scheduler,
    losses=losses,
    optimizer=optimizer,
    show_progress_bar=True,  # disable progress bar for better vis in sphinx gallery.
    train_dataloader=train_dataloader,
    eval_dataloader=val_dataloader,
    plot_images=False, # You can plot images, to see the network's actual denoising. Significantly increases the size of the notebook.
    plot_measurements=True,
    online_measurements=True,  # Do our datasets contain both (x_i, y_i) or we want to generate y_i = physics(x_i) on the fly?
    no_learning_method = "y", # Baseline to compare with: options 'y', 'A_dagger', 'A_adjoint', 'prox_l2'
    ckp_interval = 20, # Store network parameters every ckp_interval epochs. Very useful in practice.
)

In [ ]:
trainer.train()

In [ ]:
model = trainer.load_best_model()

In [ ]:
print(solver.reg.alphaj)
print(solver.reg.alpha)
print(solver.reg.beta)

In [ ]:
# load model with best validation PSNR
torch.cuda.empty_cache()


# model.reg.set_image_size(320)
trainer.plot_images = True
# trainer.test(test_dataloader)

model.reg.set_image_size(320)
trainer.test(test_dataloader)

# Nonconvex FoE
Now, we use a nonconvex Field-of-Experts model. We no longer want to update alpha since we want it to be < 1. To avoid extra training, we load the convex model from above as initialization. 

In [ ]:
if grayscale:
    reg_nc = FoE(
        in_channels=1, num_filters=64, kernel_dim = 5, device = device, conv = False, beta = 1e3, img_size = img_size, norm_weights = True, alpha = 1e0, alpha_learnable = False
    )
    solver_nc = Solver(100,5,reg_nc, init='pinv', alpha_decay = 0.87, tr = 1e-6)
    checkpoint = torch.load(BASE_DIR / "ckpts" / "denoising" / '26-02-10-14:39:44'/ 'ckp_best.pth.tar', map_location=device, weights_only=True)
else:
    reg_nc = FoE(
        in_channels=3, num_filters=64, kernel_dim = 5, device = device, conv = False, beta = 1e2, img_size = img_size, norm_weights = True, alpha = 1e0, alpha_learnable = False
    )
    solver_nc = Solver(100,5,reg_nc, init='pinv', alpha_decay = 0.87)
    checkpoint = torch.load(BASE_DIR / "ckpts" / "denoising" / '26-02-10-14:52:21'/ 'ckp_best.pth.tar', map_location=device, weights_only=True)
    
solver_nc.load_state_dict(checkpoint["state_dict"], strict=True)
solver_nc.reg.conv = False
solver_nc.reg.alpha.data = solver_nc.reg.alpha.data.clamp(max = 1)
# solver_nc.reg.alpha = solver_nc.reg.alpha.data.clone().detach()

print(solver_nc.reg.alphaj)
print(solver_nc.reg.alpha)


In [ ]:
wandb.run.finish()

In [ ]:
torch.cuda.empty_cache()

verbose = True  # print training information
wandb_vis = False  # plot curves and images in Weight&Bias

epochs = 50 # choose training epochs

learning_rate = 1e-3#1e-3

# choose training losses
losses = [dinv.loss.SupLoss(metric=dinv.metric.MSE())]#, ZeroSumLoss()]

# choose optimizer and scheduler
optimizer = torch.optim.Adam(solver_nc.reg.parameters(), lr=learning_rate, weight_decay=1e-8)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer,step_size=epochs,gamma=0.99) #step_size=1, gamma=0.1**(1.0/epochs))


if save_to_wandb:
    wandb_vis = True  # plot curves and images in Weight&Bias
    wandb_setup = {
        # Set the wandb entity where your project will be logged (generally your team name).
        "entity" : "oleh-melnyk-technical-university-of-berlin",
        # Set the wandb project where this run will be logged.
        "project" : "FoE-training",
        # Track hyperparameters and run metadata.
        "config" : {
            "learning_rate": learning_rate,
            "architecture": "FoE",
            "dataset": "CBSD68",
            "epochs": epochs,
            "scheduler_step_size": epochs,
            "sigma" : sigma,
            "alpha" : reg_nc.alpha.cpu().item(),
            "kernel_dim" : reg.kernel_size,
            "num_filters" : reg.n_filters,
            "weight_decay" : 1e-8,
            "T" : solver_nc.T,
            "K" : solver_nc.K,
            "img_size" : img_size,
            "norm_weights": reg_nc.norm_weights,
            "init" : solver_nc.init,
            "conv" : solver_nc.reg.conv,
            "beta" : solver_nc.reg.beta.cpu().item(),
            "grayscale" : grayscale,
            "tr" : solver_nc.tr,
        },
    }
else:
    wandb_vis = False
    wandb_setup = None

trainer_nc = Bilevel_Trainer(
    solver_nc,
    device=device,
    save_path=str(CKPT_DIR / operation),
    verbose=verbose,
    wandb_vis=wandb_vis,
    wandb_setup = wandb_setup,
    physics=physics,
    epochs=epochs,
    scheduler=scheduler,
    losses=losses,
    optimizer=optimizer,
    show_progress_bar=True,  # disable progress bar for better vis in sphinx gallery.
    train_dataloader=train_dataloader,
    eval_dataloader=val_dataloader,
    plot_images=False, # You can plot images, to see the network's actual denoising. Significantly increases the size of the notebook.
    plot_measurements=True,
    online_measurements=True,  # Do our datasets contain both (x_i, y_i) or we want to generate y_i = physics(x_i) on the fly?
    no_learning_method = "y", # Baseline to compare with: options 'y', 'A_dagger', 'A_adjoint', 'prox_l2'
    ckp_interval = 20, # Store network parameters every ckp_interval epochs. Very useful in practice.
)

In [ ]:
trainer_nc.train()

In [ ]:
# load model with best validation PSNR
model = trainer_nc.load_best_model()

print(solver_nc.reg.alphaj)
print(solver_nc.reg.alpha)
print(solver_nc.reg.beta)

model.reg.set_image_size(320)
trainer_nc.plot_images = True
trainer_nc.test(test_dataloader)

# Plot an example of reconstruction

In [ ]:
from deepinv.utils.plotting import plot_inset, plot

def show_image_comparison(images, suptitle=None, ref=None):
    """Display various images restoration with PSNR and zoom-in"""

    titles = list(images.keys())
    if "Original" in images or ref is not None:
        # If the original image is in the dict, add PSNR in the titles.
        image = images["Original"] if "Original" in images else ref
        psnr = [dinv.metric.cal_psnr(image, im).item() for im in images.values()]
        titles = [
            f"{name} \n (PSNR: {psnr:.2f})" if name != "Original" else name
            for name, psnr in zip(images.keys(), psnr)
        ]
    # Plot the images with zoom-in
    fig = plot_inset(
        list(images.values()),
        titles=titles,
        extract_size=0.2,
        extract_loc=(0.5, 0.0),
        inset_size=0.5,
        return_fig=True,
        show=False,
        figsize=(len(images) * 1.5, 2.5),
    )

    # Add a suptitle if it is provided
    if suptitle:
        plt.suptitle(suptitle, size=12)
        plt.tight_layout()
        fig.subplots_adjust(top=0.85, bottom=0.02, left=0.02, right=0.95)
        plt.show()

In [ ]:
# load convex FoE
if grayscale:
    reg = FoE(in_channels=1, num_filters=64, kernel_dim = 5, device = device, conv = True, beta = 1e3, img_size = img_size, norm_weights = True, alpha = 1e0, alpha_learnable = True)
else:
    reg = FoE(in_channels=3, num_filters=64, kernel_dim = 5, device = device, conv = True, beta = 1e3, img_size = img_size, norm_weights = True, alpha = 1e0, alpha_learnable = True)
solver = Solver(50,5,reg, init='pinv', tr = 1e-4)

if grayscale:
    checkpoint = torch.load(BASE_DIR / "ckpts" / "denoising" / '26-02-10-14:39:44'/ 'ckp_best.pth.tar', map_location=device, weights_only=True)
else:
    checkpoint = torch.load(BASE_DIR / "ckpts" / "denoising" / '26-02-10-14:52:21'/ 'ckp_best.pth.tar', map_location=device, weights_only=True)
    
solver.load_state_dict(checkpoint["state_dict"], strict=True)
solver.eval()
solver.physics_spectral = 1.0
solver.reg.set_image_size(320)


if grayscale:
    reg_nc = FoE(in_channels=1, num_filters=64, kernel_dim = 5, device = device, conv = False, beta = 1e3, img_size = img_size, norm_weights = True, alpha = 1e0, alpha_learnable = False)
else:
    reg_nc = FoE(in_channels=3, num_filters=64, kernel_dim = 5, device = device, conv = False, beta = 1e3, img_size = img_size, norm_weights = True, alpha = 1e0, alpha_learnable = False)
    
solver_nc = Solver(50,5,reg_nc, init='pinv', tr = 1e-6)
if grayscale:
    checkpoint = torch.load(BASE_DIR / "ckpts" / "denoising" / '26-02-10-14:43:13'/ 'ckp_best.pth.tar', map_location=device, weights_only=True)
else:
    checkpoint = torch.load(BASE_DIR / "ckpts" / "denoising" / '26-02-10-15:00:39'/ 'ckp_best.pth.tar', map_location=device, weights_only=True)
solver_nc.load_state_dict(checkpoint["state_dict"], strict=True)
solver_nc.eval()
solver_nc.physics_spectral = 1.0
solver_nc.reg.set_image_size(320)


bm3d = dinv.models.BM3D()
if grayscale:
    drunet = dinv.models.DRUNet(in_channels =1, out_channels=1).to(device)
else:
    drunet = dinv.models.DRUNet().to(device)

x = test_dataset[41].to(device)
x = x[None,...]
y = physics(x)

solver.eval()
solver.reg.set_image_size(320)
solver_nc.eval()
solver_nc.reg.set_image_size(320)

denoiser_results = {
    "Original": x,
    "Noisy": y,
    "BM3D": bm3d(y, sigma),
    "FoE (Conv)": solver(y,physics),
    "FoE (Nonconv)": solver_nc(y,physics),
    "DRUNet": drunet(y, sigma),
}
show_image_comparison(denoiser_results, suptitle=rf"Noise level $\sigma={sigma:.2f}$")

# Plot learned filters

In [ ]:
def plot_filters(reg, idx = 0, title = ''):
    with torch.no_grad():
        filters = torch.zeros([4,3,13,13]).to(device)
        filters[:,:,4:9,4:9] = reg.W[0].weight.clone().detach()
        filters = filters.swapaxes(0,1)
        # print(filters.shape)
        filters = reg.W[1](filters)
        filters = reg.W[2](filters)
        # print(filters.shape)
        filters = filters.swapaxes(0,1)

        filters_stacked = torch.zeros([3,13*8,13*8]).to(device)

        for i in range(8):
            for j in range(8):
                filters_stacked[:,(i*13):((i+1)*13),(j*13):((j+1)*13)] = filters[i*8 + j,:, :,:]

        vmin = filters_stacked.min().item()
        vmax = filters_stacked.max().item()
        filters_stacked = filters_stacked 
        plt.imshow(filters_stacked[idx].cpu(), interpolation='none', cmap = 'Greys')
        plt.title(title)
        plt.colorbar()
        plt.axis("off")
        plt.show()

if grayscale:
    plot_filters(solver.reg, 0, 'FoE Conv')
    plot_filters(solver_nc.reg, 0, 'FoE Nonconv')
else:
    plot_filters(solver.reg,0, 'FoE Conv 0')
    plot_filters(solver.reg,1, 'FoE Conv 1')
    plot_filters(solver.reg,2, 'FoE Conv 2')
    plot_filters(solver_nc.reg,0, 'FoE Nonconv 0')
    plot_filters(solver_nc.reg,1, 'FoE Nonconv 1')
    plot_filters(solver_nc.reg,2, 'FoE Nonconv 2')

# Test on CT reconstruction

In [ ]:
n_channels = 1  # 3 for color images, 1 for gray-scale images
img_size = 64

operation = "tomography"
method = "learned_regularizer"
measurement_dir = DATA_DIR / DATASET_DIR / operation

physics_ct = dinv.physics.Tomography(
    img_width=img_size,
    angles=15, # in degrees
    normalize=False,
    device=device,
    noise_model=dinv.physics.GaussianNoise(sigma=sigma), # optional complication
)

In [ ]:
if grayscale:
    test_transform = transforms.Compose(
        [v2.Grayscale(), transforms.CenterCrop(img_size), transforms.ToTensor()]
    )
else:
    test_transform = transforms.Compose(
        [transforms.CenterCrop(img_size), transforms.ToTensor()]
    )


val_dataset = dinv.datasets.BSDS500(DATASET_DIR, download=True, train = True, transform=test_transform)
val_size = 5
val_dataset = torch.utils.data.Subset(val_dataset, indices[-val_size:])

test_dataset = dinv.datasets.CBSD68(root = DATASET_DIR_CBSD, download=True, transform = test_transform, rotate = False)
indices_test = torch.randperm(len(test_dataset))
# CBSD68 is stored as hugging face dataset and we need to use their subsampling function insteado f torch.utils.data.Subset
test_dataset.hf_dataset = test_dataset.hf_dataset.take(val_size)

val_dataloader = DataLoader(
    val_dataset, batch_size=1, num_workers=num_workers, shuffle=False
)

test_dataloader = DataLoader(
    test_dataset, batch_size=1, num_workers=num_workers, shuffle=False
)

In [ ]:
from deepinv.training import test
from deepinv.optim.optimizers import optim_builder
from deepinv.optim.data_fidelity import L2
import numpy as np

early_stop = True  # Stop algorithm when convergence criteria is reached
backtracking = False
use_bicubic_init = False  # Use bicubic interpolation to initialize the algorithm
batch_size = 1  # batch size for evaluation is necessarily 1 for early stopping and backtracking to work.
max_iter = 500

# Select the data fidelity term
data_fidelity = L2()

def param_search(label, model_func, param_range):
    best_PSNR = -99
    best_par = param_range[0]
    best_ref = -1
    RESULTS_DIR = BASE_DIR / label

    # Select the best model on the validation set
    for ni in range(len(param_range)):
        par = param_range[ni]
        model = model_func(par)

        res = test(
            model=model,
          test_dataloader=val_dataloader,
          physics=physics_ct,
          device=device,
          plot_images=False,
          save_folder=RESULTS_DIR / method / operation / measurement_dir / str(ni),
          plot_convergence_metrics=False,
          verbose=True,
          no_learning_method = "A_dagger",
          online_measurements=True,  
        )
        print('Parameter:', par, ' PSNR: ', res["PSNR"])
        
        if res["PSNR"] > best_PSNR:
            best_PSNR = res["PSNR"]
            best_par = par

    # Use the best model on the test set
    model = model_func(best_par)
    RESULTS_DIR = BASE_DIR / label / 'best'

    best_res = test(
      model=model,
      test_dataloader=test_dataloader,
      physics=physics_ct,
      device=device,
      plot_images=True,
      save_folder=RESULTS_DIR / method / operation / measurement_dir / 'best',
      plot_convergence_metrics=False,
      verbose=True,
      no_learning_method = "A_dagger",
      online_measurements=True,
    )
    
    print('Best: Paramereter: ', best_par, 'Test PSNR: ', best_res["PSNR"])
    return model, best_par, best_res["PSNR"]

# Use TV regularization as baseline

In [ ]:
def prepare_TV_model(reg):
    params_algo = {
                "stepsize": 1.0,
                "lambda": reg,
                "beta": 1.0,
                }

    model_tv = optim_builder(
        iteration="ADMM",
        prior=dinv.optim.prior.TVPrior(n_it_max=20), # Use TV prior with 20 subitertations
        data_fidelity=data_fidelity,
        early_stop=early_stop,
        max_iter=max_iter,
        verbose=False,
        params_algo=params_algo,
    )
    model_tv.eval()
    return model_tv

In [ ]:
model_tv, reg_tv_best, tv_psnr = param_search('TV', prepare_TV_model, 10**np.arange(-2,0.1,0.5))

In [ ]:
def prepare_FoE(par, conv, grayscale, spec_norm, num_it, img_size):
    if conv and grayscale:
        reg = FoE(in_channels=1, num_filters=64, kernel_dim = 5, device = device, conv = True, beta = 1e3, img_size = img_size, norm_weights = True, alpha = 1e0, alpha_learnable = True)
        solver = Solver(int(num_it),5,reg, init='pinv', tr = 1e-4)
        checkpoint = torch.load(BASE_DIR / "ckpts" / "denoising" / '26-02-10-14:39:44'/ 'ckp_best.pth.tar', map_location=device, weights_only=True)
    elif conv and not grayscale:
        reg = FoE(in_channels=3, num_filters=64, kernel_dim = 5, device = device, conv = True, beta = 1e2, img_size = img_size, norm_weights = True, alpha = 1e0, alpha_learnable = True)
        solver = Solver(int(num_it),5,reg, init='pinv', tr = 1e-4)
        checkpoint = torch.load(BASE_DIR / "ckpts" / "denoising" / '26-02-10-14:52:21'/ 'ckp_best.pth.tar', map_location=device, weights_only=True)
    elif not conv and grayscale:
        reg = FoE(in_channels=1, num_filters=64, kernel_dim = 5, device = device, conv = False, beta = 1e3, img_size = img_size, norm_weights = True, alpha = 1e0, alpha_learnable = False)
        solver = Solver(int(num_it),5,reg, init='pinv', tr = 1e-6)
        checkpoint = torch.load(BASE_DIR / "ckpts" / "denoising" / '26-02-10-14:43:13'/ 'ckp_best.pth.tar', map_location=device, weights_only=True) 
    else:
        reg = FoE(in_channels=3, num_filters=64, kernel_dim = 5, device = device, conv = False, beta = 1e2, img_size = img_size, norm_weights = True, alpha = 1e0, alpha_learnable = False)
        solver = Solver(int(num_it),5,reg, init='pinv', tr = 1e-6)
        checkpoint = torch.load(BASE_DIR / "ckpts" / "denoising" / '26-02-10-15:00:39'/ 'ckp_best.pth.tar', map_location=device, weights_only=True)    
        
    solver.load_state_dict(checkpoint["state_dict"], strict=True)
    solver.eval()
    solver.reg.set_image_size(img_size)
    solver.reg.alpha.data = par * torch.ones([1]).to(device)
    solver.physics_spectral = spec_norm
    
    return solver

# Compare with FoE

In [ ]:
spec_norm = physics_ct.compute_norm(
                torch.randn(
                    (1, img_size, img_size),
                    generator=torch.Generator(device).manual_seed(0),
                    device=device,
                )[None],
                squared=False,
                verbose=False,
            )
print(spec_norm)
prep_conv = lambda par: prepare_FoE(par, True, True, spec_norm = spec_norm.item(), num_it = 200, img_size = img_size)
model_FoEc, reg_FoEc_best, FoEc_psnr = param_search('FoE_conv', prep_conv, 10**np.arange(-6,3,0.5))

In [ ]:
prep_nonconv = lambda par: prepare_FoE(par, False, True, spec_norm.item(), num_it = 200, img_size = img_size)
model_FoE, reg_FoE_best, FoE_psnr = param_search('FoE', prep_nonconv, 10**np.arange(-4,3,0.5))